### Intel Lab Data

[https://db.csail.mit.edu/labdata/labdata.html]

`mote_locs.txt` contains the locations: id, x, y. 
`data` ontains 2.3 million readings from sensors. Data is in the form: 
- date: yyyy-mm-dd
- time: hh:mm:ss.xxx
- epoch: int
- moteid: int
- temperature: real (Celsius)
- humidity: real
- light: real
- voltage: real


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

In [ ]:
position_data = pd.read_table('mote_locs.txt', names = ['mote_id', 'x', 'y'], sep=' ')
position_data.head()

In [ ]:
plt.scatter(position_data['x'], position_data['y'])
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.title('Mote Locations')

plt.show()

In [ ]:
### Read data and preprocess
data = pd.read_table('data', names = ['date', 'time', 'epoch', 'mote_id', 'temperature', 'humidity', 'light', 'voltage'], sep=' ')

print(data.info())

# Convert 'date' and 'time' columns to seconds since start of data collection
data['datetime'] = pd.to_datetime(data['date'] + ' ' + data['time'], format='mixed')
data['seconds_since_start'] = (data['datetime'] - data['datetime'].min()).dt.total_seconds()

# Drop unnecessary columns
data.drop(columns=['light', 'humidity', 'voltage', 'date', 'time', 'datetime'], inplace=True)

# only keep the first NT time steps for the animation
NT = 1000
data = data[data["epoch"] < NT]

data["seconds_since_start"] = (np.floor(data["seconds_since_start"]).astype("int32"))
data.head()

In [ ]:
data.to_feather("intel_data_processed.feather", compression="zstd")

In [ ]:
## histogram of seconds_since_start
plt.hist(data["seconds_since_start"], bins=50)
plt.xlabel("Seconds Since Start")
plt.ylabel("Frequency")
plt.title("Histogram of Seconds Since Start")
plt.show()

In [ ]:


# ---------------------------------------------------------
# 1. Combine temperature measurements with sensor positions
# ---------------------------------------------------------
df = data.merge(position_data, on="mote_id", how="left")






# Sort for convenience
df = df.sort_values(["epoch", "mote_id"])

# All unique time steps
epochs = np.sort(df["epoch"].unique())

# ---------------------------------------------------------
# 2. Set up plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

# Use a fixed color scale across the entire animation.
# This is important so that colors mean the same thing
# at every time step.
vmin = df["temperature"].min()
vmax = df["temperature"].max()

# Data at first time step
first = df[df["epoch"] == epochs[0]]

scatter = ax.scatter(
    first["x"],
    first["y"],
    c=first["temperature"],
    cmap="coolwarm",
    vmin=vmin,
    vmax=vmax,
    s=100,
    edgecolors="black"
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Temperature")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Epoch: {epochs[0]}")

# Keep spatial extent fixed
ax.set_xlim(df["x"].min() - 0.5, df["x"].max() + 0.5)
ax.set_ylim(df["y"].min() - 0.5, df["y"].max() + 0.5)

ax.set_aspect("equal")


# ---------------------------------------------------------
# 3. Animation update function
# ---------------------------------------------------------
def update(frame):
    epoch = epochs[frame]

    current = df[df["epoch"] == epoch]

    # Update sensor positions
    scatter.set_offsets(
        current[["x", "y"]].to_numpy()
    )

    # Update temperatures/colors
    scatter.set_array(
        current["temperature"].to_numpy()
    )

    ax.set_title(f"Epoch: {epoch}")

    return scatter,


# ---------------------------------------------------------
# 4. Create animation
# ---------------------------------------------------------
anim = FuncAnimation(
    fig,
    update,
    frames=len(epochs),
    interval=100,   # milliseconds between frames
    blit=False
)

plt.show()

In [ ]:
# anim.save(
#     "temperature_animation.gif",
#     writer="pillow",
#     fps=10
# )